# BioLogic Echem — Rate Test Demo

This notebook demonstrates the full workflow for analysing galvanostatic charge-discharge (GCPL / rate test) data exported from BioLogic EC-Lab.

**What you will learn:**
1. Load a `.xlsx` EC-Lab export
2. Detect half-cycles automatically from current sign
3. Extract a specific cycle and compute specific capacity (mAh g⁻¹)
4. Plot Voltage vs. Capacity
5. Overlay multiple C-rates on one figure
6. Compute and plot differential capacity (dQ/dV)
7. Export figures and summary CSV

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))  # repo root

import matplotlib.pyplot as plt
import pandas as pd

from biologic_echem import (
    load_xlsx,
    segment_halfcycles,
    get_cycle,
    specific_capacity,
    coulombic_efficiency,
    dqdv,
    discover_rate_files,
    new_figure,
    plot_vq,
    plot_dqdv,
    style_axes,
    DEFAULT_DPI,
)

DATA_DIR = pathlib.Path('../data')
OUT_DIR  = pathlib.Path('../output')
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'figures').mkdir(exist_ok=True)
(OUT_DIR / 'csv').mkdir(exist_ok=True)

print('Package loaded OK')

## 1 · Single-file analysis

Load one rate-test file, segment into half-cycles, extract cycle 2, and plot.

In [ ]:
# ── User settings ─────────────────────────────────────────────────────────
FILE     = DATA_DIR / '1C.xlsx'
MASS_MG  = 0.185   # active material mass (mg)
CYCLE_N  = 2       # which full cycle to analyse (1-indexed)
V_LIM    = (1.0, 3.0)   # voltage window for plot
# ──────────────────────────────────────────────────────────────────────────

df = load_xlsx(FILE)
print(f'Loaded {len(df):,} data points from {FILE.name}')
df.head()

In [ ]:
halves = segment_halfcycles(df)
print(f'Found {len(halves)} half-cycles  →  {len(halves)//2} complete cycles')

chg, dis = get_cycle(halves, n=CYCLE_N)

Q_chg = specific_capacity(chg, MASS_MG)
Q_dis = specific_capacity(dis, MASS_MG)

CE = coulombic_efficiency(Q_chg, Q_dis)
print(f'Cycle {CYCLE_N}:  Q_chg = {Q_chg[-1]:.1f} mAh/g,  '
      f'Q_dis = {Q_dis[-1]:.1f} mAh/g,  CE = {CE*100:.1f} %')

In [ ]:
fig, ax = new_figure()
plot_vq(ax, Q_chg, chg['E'].values, Q_dis, dis['E'].values)
style_axes(ax, title=f'1C — Cycle {CYCLE_N}', v_lim=V_LIM)
plt.tight_layout()
plt.savefig(OUT_DIR / 'figures' / '1C_cycle2.png', dpi=DEFAULT_DPI, bbox_inches='tight')
plt.show()

## 2 · Multi-rate comparison

Overlay the discharge curve from each C-rate file on a single figure.

In [ ]:
# ── User settings ─────────────────────────────────────────────────────────
MASS_MG      = 0.185
CYCLE_N      = 2
V_LIM        = (1.0, 3.0)
DISCHARGE_ONLY = True   # set False to show both charge and discharge
# ──────────────────────────────────────────────────────────────────────────

rate_files = discover_rate_files(DATA_DIR)
print('Files found:')
for lbl, fp in rate_files:
    print(f'  {lbl:12s}  {fp.name}')

In [ ]:
fig, ax = new_figure(figsize=(7, 5))
summary_rows = []

for label, fp in rate_files:
    df = load_xlsx(fp)
    halves = segment_halfcycles(df)
    try:
        chg, dis = get_cycle(halves, n=CYCLE_N)
    except ValueError as err:
        print(f'  Skipping {label}: {err}')
        continue

    Q_chg = specific_capacity(chg, MASS_MG)
    Q_dis = specific_capacity(dis, MASS_MG)
    CE    = coulombic_efficiency(Q_chg, Q_dis)

    if DISCHARGE_ONLY:
        ax.plot(Q_dis, dis['E'].values, lw=1.5, label=label)
    else:
        plot_vq(ax, Q_chg, chg['E'].values, Q_dis, dis['E'].values, label=label)

    summary_rows.append({
        'rate':          label,
        'Q_chg_mAhg':    round(Q_chg[-1], 2),
        'Q_dis_mAhg':    round(Q_dis[-1], 2),
        'CE_pct':        round(CE * 100, 2),
    })

style_axes(ax, title=f'Rate Performance — Cycle {CYCLE_N} Discharge', v_lim=V_LIM)
ax.set_xlabel('Specific Capacity (mAh g⁻¹)')
ax.set_ylabel('Voltage (V)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'figures' / 'rate_comparison.png', dpi=DEFAULT_DPI, bbox_inches='tight')
plt.show()

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / 'csv' / 'rate_summary.csv', index=False)
print('\nSummary:')
summary

## 3 · Differential capacity (dQ/dV)

Peaks in the dQ/dV plot correspond to electrochemical phase transitions.

In [ ]:
# ── User settings ─────────────────────────────────────────────────────────
FILE    = DATA_DIR / '1C.xlsx'
MASS_MG = 0.185
CYCLE_N = 2
V_LIM   = (1.0, 3.0)
DV      = 0.005   # voltage grid spacing (V)
SIGMA   = 2.0     # Gaussian smoothing sigma (V)
# ──────────────────────────────────────────────────────────────────────────

df = load_xlsx(FILE)
halves = segment_halfcycles(df)
chg, dis = get_cycle(halves, n=CYCLE_N)

Q_chg = specific_capacity(chg, MASS_MG)
Q_dis = specific_capacity(dis, MASS_MG)

V_chg_grid, dQdV_chg = dqdv(Q_chg, chg['E'].values, dv=DV, sigma=SIGMA)
V_dis_grid, dQdV_dis = dqdv(Q_dis, dis['E'].values, dv=DV, sigma=SIGMA)

fig, ax = new_figure()
plot_dqdv(ax, V_chg_grid, dQdV_chg, V_dis_grid, dQdV_dis)
style_axes(ax, title=f'Differential Capacity — 1C, Cycle {CYCLE_N}', v_lim=V_LIM)
plt.tight_layout()
plt.savefig(OUT_DIR / 'figures' / '1C_dqdv.png', dpi=DEFAULT_DPI, bbox_inches='tight')
plt.show()

## 4 · Export processed data to CSV

In [ ]:
import numpy as np

for label, fp in discover_rate_files(DATA_DIR):
    df = load_xlsx(fp)
    halves = segment_halfcycles(df)
    try:
        chg, dis = get_cycle(halves, n=2)
    except ValueError:
        continue

    Q_chg = specific_capacity(chg, MASS_MG)
    Q_dis = specific_capacity(dis, MASS_MG)

    out = pd.DataFrame({
        'Q_chg_mAhg':  Q_chg,
        'V_chg':       chg['E'].values,
    }).join(
        pd.DataFrame({
            'Q_dis_mAhg': Q_dis,
            'V_dis':      dis['E'].values,
        }), how='outer'
    )

    fname = OUT_DIR / 'csv' / f'{label}_cycle2.csv'
    out.to_csv(fname, index=False)
    print(f'Saved {fname.name}')